# 02 · Correlation & driver analysis
Cross-KPI, lead/lag, sibling-site, layer variance split.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 60)
from networkanalysis.db.database import query_df, table_counts
from networkanalysis.pipeline.features import build_site_feature_table, KPI_DIRECTION, HEADLINE_KPIS

In [ ]:
from networkanalysis.analytics import correlation as corr
from networkanalysis.analytics import scoring
feat = build_site_feature_table()
sc, _ = scoring.compute_scorecard(feat)

In [ ]:
# market-wide correlation matrix of the key metrics
cols = ["tcp_client_rtt_ms","tcp_server_rtt_ms","dl_throughput_mbps","vonr_mos","youtube_qoe_mos",
        "path_delay_ms","twamp_loss_pct","sevone_queue_depth","prb_util_p95","rsrq_p50"]
C = feat[cols].corr(method="spearman")
plt.figure(figsize=(8,6)); plt.imshow(C, cmap="coolwarm", vmin=-1, vmax=1)
plt.xticks(range(len(cols)), cols, rotation=90); plt.yticks(range(len(cols)), cols); plt.colorbar(); plt.title("Spearman")

In [ ]:
corr.compute_correlations(feat, sc).query("scope=='market'")

In [ ]:
# lead/lag: does SevOne queue buildup precede the YouTube QoE drop?
d = feat.groupby("ts_hour")[["sevone_queue_depth","youtube_qoe_mos"]].mean()
lags = range(-6,7)
cc = [d.sevone_queue_depth.corr(d.youtube_qoe_mos.shift(-L)) for L in lags]
plt.plot(list(lags), cc, marker="o"); plt.axvline(0, ls=":"); plt.xlabel("lag (hours)"); plt.ylabel("corr")
plt.title("queue depth vs YouTube MOS cross-correlation")

In [ ]:
corr.layer_driver_split(feat, sc).describe()